In [31]:
"""
RBS Feature Engineering - Fixed Version
修复内容：
1. 允许SD基序有1-2个错配
2. 新增RBS强度对数变换
3. 新增SD区域特征
4. 新增间隔区特征
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

import sys

# 自动获取当前文件所在目录，并切换到项目根目录
# 由于是 Jupyter Notebook，__file__ 不可用，使用 os.getcwd() 和 os.path
current_dir = os.getcwd()
print(f"当前工作目录: {current_dir}")

# 向上查找 feature_engineering 项目根目录
project_root = current_dir
while project_root:
    if os.path.basename(project_root) == 'feature_engineering':
        break
    parent = os.path.dirname(project_root)
    if parent == project_root:  # 到达磁盘根目录
        break
    project_root = parent

# 如果找到了项目根目录，切换到该目录
if os.path.basename(project_root) == 'feature_engineering':
    os.chdir(project_root)
    print(f"✅ 切换到项目根目录: {os.getcwd()}")
else:
    print(f"⚠️ 未找到项目根目录，当前目录: {os.getcwd()}")

当前工作目录: c:\Users\16926\Desktop\AI+合成生物创新大赛\feature_engineering
✅ 切换到项目根目录: c:\Users\16926\Desktop\AI+合成生物创新大赛\feature_engineering


In [32]:
# ============================================================
# OUTPUT DIRECTORY
# ============================================================
OUTPUT_DIR = '../feature_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/plots', exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/data', exist_ok=True)

print("=" * 70)
print("RBS FEATURE ENGINEERING - FIXED VERSION")
print("=" * 70)

RBS FEATURE ENGINEERING - FIXED VERSION


In [ ]:
# ============================================================
# 1. LOAD DATA
# ============================================================
rbss = pd.read_excel('RBS_data.xlsx', engine='openpyxl')
print(f"Loaded {len(rbss)} RBS sequences")

FileNotFoundError: [Errno 2] No such file or directory: 'RBS_data.xlsx'

In [36]:
# ============================================================
# 2. RBS FEATURE EXTRACTION FUNCTIONS (FIXED)
# ============================================================

def gc_content(seq):
    """Calculate GC content"""
    seq = str(seq).replace(' ', '')
    if len(seq) == 0:
        return 0
    return (seq.count('G') + seq.count('C')) / len(seq)

def at_content(seq):
    """Calculate AT content"""
    seq = str(seq).replace(' ', '')
    if len(seq) == 0:
        return 0
    return (seq.count('A') + seq.count('T')) / len(seq)

def gc_skew(seq):
    """GC skew: (G-C)/(G+C)"""
    seq = str(seq).replace(' ', '')
    g = seq.count('G')
    c = seq.count('C')
    if g + c == 0:
        return 0
    return (g - c) / (g + c)

def hamming_distance(seq1, seq2):
    """Calculate Hamming distance"""
    min_len = min(len(seq1), len(seq2))
    return sum(1 for i in range(min_len) if seq1[i] != seq2[i])

def find_sd_with_mismatch(seq, motif='AGGAGG', max_mismatch=2):
    """
    Find SD motif with allowed mismatches
    Returns: (start_position, mismatch_count) or (None, None)
    """
    seq = str(seq).replace(' ', '')
    motif_len = len(motif)
    
    for i in range(len(seq) - motif_len + 1):
        mismatches = sum(1 for a, b in zip(seq[i:i+motif_len], motif) if a != b)
        if mismatches <= max_mismatch:
            return i, mismatches
    return None, None

def calculate_sd_score(seq, motif='AGGAGG'):
    """
    Calculate SD sequence quality score
    Higher = better match to consensus
    """
    seq = str(seq).replace(' ', '')
    motif_len = len(motif)
    best_score = 0
    
    for i in range(len(seq) - motif_len + 1):
        matches = sum(1 for a, b in zip(seq[i:i+motif_len], motif) if a == b)
        score = matches / motif_len
        if score > best_score:
            best_score = score
    
    return best_score

def calculate_rbs_mfe(seq):
    """
    Approximate minimum free energy for RBS
    """
    seq = str(seq).replace(' ', '')
    # 简化的MFE：GC含量越高越稳定
    gc = gc_content(seq)
    return -gc * 12  # RBS通常较短，系数略低

def sequence_complexity(seq):
    """Sequence complexity"""
    seq = str(seq).replace(' ', '')
    if len(seq) < 3:
        return 0
    kmers = set(seq[i:i+3] for i in range(len(seq)-2))
    return len(kmers) / (len(seq) - 2)

def rbs_specific_features(seq):
    """Extract RBS-specific features - FIXED"""
    seq = str(seq).replace(' ', '')
    original_seq = seq
    
    # Remove CATATG (start codon) for SD analysis
    rbs_seq = seq[:-6] if seq.endswith('CATATG') else seq
    
    features = {}
    
    # ===== SD MOTIF DETECTION (FIXED) =====
    # Search with 0, 1, 2 mismatches
    sd_pos, sd_mismatches = find_sd_with_mismatch(rbs_seq, 'AGGAGG', max_mismatch=2)
    
    if sd_pos is not None:
        features['sd_present'] = 1
        features['sd_mismatches'] = sd_mismatches
        features['sd_start_pos'] = sd_pos
        features['sd_end_pos'] = sd_pos + 6
        
        # SD sequence itself
        sd_sequence = rbs_seq[sd_pos:sd_pos+6]
        features['sd_sequence_score'] = 6 - sd_mismatches  # 6 = perfect match
        
        # Distance from SD end to start codon (CATATG)
        # start codon is at end of rbs_seq
        start_codon_pos = len(rbs_seq)
        features['sd_to_start_distance'] = start_codon_pos - (sd_pos + 6)
        
        # Spacer between SD and start
        spacer = rbs_seq[sd_pos+6:start_codon_pos]
        features['spacer_length'] = len(spacer)
        features['spacer_gc'] = gc_content(spacer) if len(spacer) > 0 else 0
        features['spacer_at'] = at_content(spacer) if len(spacer) > 0 else 0
        
        # A-rich region before start (last 15 bases before start)
        before_start = rbs_seq[-15:] if len(rbs_seq) >= 15 else rbs_seq
        features['a_rich_before_start'] = at_content(before_start)
        
    else:
        # No SD motif found
        features['sd_present'] = 0
        features['sd_mismatches'] = 999
        features['sd_start_pos'] = 999
        features['sd_end_pos'] = 999
        features['sd_sequence_score'] = 0
        features['sd_to_start_distance'] = 999
        features['spacer_length'] = 999
        features['spacer_gc'] = 0
        features['spacer_at'] = 0
        features['a_rich_before_start'] = at_content(rbs_seq[-15:]) if len(rbs_seq) >= 15 else at_content(rbs_seq)
    
    # ===== SD SCORE (best match anywhere) =====
    features['sd_best_score'] = calculate_sd_score(rbs_seq, 'AGGAGG')
    
    # ===== BASIC SEQUENCE FEATURES =====
    features['rbs_gc'] = gc_content(rbs_seq)
    features['rbs_at'] = at_content(rbs_seq)
    features['rbs_gc_skew'] = gc_skew(rbs_seq)
    features['rbs_length'] = len(rbs_seq)
    features['rbs_complexity'] = sequence_complexity(rbs_seq)
    features['rbs_mfe'] = calculate_rbs_mfe(rbs_seq)
    
    # ===== SD ENRICHMENT =====
    # Count partial SD motifs
    features['sd_partial_count'] = (rbs_seq.count('AGG') + 
                                     rbs_seq.count('GGA') + 
                                     rbs_seq.count('GGAG'))
    
    # Purine-rich region score (important for RBS)
    purines = sum(1 for base in rbs_seq if base in 'AG')
    features['purine_richness'] = purines / max(len(rbs_seq), 1)
    
    return features

In [ ]:
# ============================================================
# 3. EXTRACT FEATURES
# ============================================================
print("\n" + "-" * 50)
print("EXTRACTING RBS FEATURES (FIXED)")
print("-" * 50)

feature_list = []

for idx, row in rbss.iterrows():
    seq = row['Sequence']
    f = {}
    
    # Strain info
    f['strain'] = row['Strain']
    
    # RBS-specific features (FIXED)
    rbs_feat = rbs_specific_features(seq)
    f.update(rbs_feat)
    
    # Target variables
    f['F_on_OD'] = row['F-on/OD']
    f['F_on_OD_log'] = np.log10(row['F-on/OD'] + 1)  # Log transform
    f['mean_RNA'] = row['mean.RNA']
    f['mean_prot'] = row['mean.prot']
    
    feature_list.append(f)

rbs_df = pd.DataFrame(feature_list)

print("Columns in rbs_df:", rbs_df.columns.tolist())
print("Unique values in sd_present:", rbs_df['sd_present'].unique())
print("Number of sample with SD found:", (rbs_df['sd_present'] == 1).sum())


--------------------------------------------------
EXTRACTING RBS FEATURES (FIXED)
--------------------------------------------------


NameError: name 'rbss' is not defined

In [ ]:
# ============================================================
# 4. SAVE FEATURE DATA
# ============================================================
rbs_df.to_csv(f'{OUTPUT_DIR}/data/rbs_features_fixed.csv', index=False)
print(f"\n✓ Saved: {OUTPUT_DIR}/data/rbs_features_fixed.csv")

In [35]:
# ============================================================
# 5. FEATURE ANALYSIS
# ============================================================
print("\n" + "=" * 70)
print("FEATURE CORRELATION ANALYSIS (FIXED)")
print("=" * 70)

# Correlation with F_on_OD
target_col = 'F_on_OD'
exclude_cols = ['strain', 'mean_RNA', 'mean_prot', 'F_on_OD_log']
feature_cols = [c for c in rbs_df.columns if c not in exclude_cols]

correlations = []
for col in feature_cols:
    if col != target_col:
        corr = rbs_df[col].corr(rbs_df[target_col])
        if not np.isnan(corr):
            correlations.append((col, abs(corr)))

# Sort by absolute correlation
correlations.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop 10 Features by Correlation with {target_col}:")
print("-" * 50)
for i, (feat, corr) in enumerate(correlations[:10], 1):
    print(f"  {i:2d}. {feat:<25} | r = {corr:.4f}")

# Correlation with log-transformed target
target_col_log = 'F_on_OD_log'
correlations_log = []
for col in feature_cols:
    if col != target_col_log:
        corr = rbs_df[col].corr(rbs_df[target_col_log])
        if not np.isnan(corr):
            correlations_log.append((col, abs(corr)))
correlations_log.sort(key=lambda x: x[1], reverse=True)

print(f"\nTop 10 Features by Correlation with {target_col_log} (log-transformed):")
print("-" * 50)
for i, (feat, corr) in enumerate(correlations_log[:10], 1):
    print(f"  {i:2d}. {feat:<25} | r = {corr:.4f}")


FEATURE CORRELATION ANALYSIS (FIXED)


NameError: name 'rbs_df' is not defined

In [ ]:
# ============================================================
# 6. SAVE CORRELATION RESULTS
# ============================================================
corr_df = pd.DataFrame(correlations, columns=['feature', 'abs_correlation'])
corr_df.to_csv(f'{OUTPUT_DIR}/data/rbs_correlations_fixed.csv', index=False)

corr_log_df = pd.DataFrame(correlations_log, columns=['feature', 'abs_correlation'])
corr_log_df.to_csv(f'{OUTPUT_DIR}/data/rbs_correlations_log_fixed.csv', index=False)

In [ ]:
# ============================================================
# 7. VISUALIZATION
# ============================================================
print("\n" + "-" * 50)
print("GENERATING VISUALIZATIONS")
print("-" * 50)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Top features bar chart
top_features = correlations[:8]
features_names = [f[0] for f in top_features]
features_corr = [f[1] for f in top_features]

axes[0].barh(features_names, features_corr, color='coral', alpha=0.8)
axes[0].set_xlabel('Absolute Correlation')
axes[0].set_title('RBS: Top 8 Features by Correlation (FIXED)')
axes[0].invert_yaxis()

# Best feature vs Strength
if len(top_features) > 0:
    best_feat = top_features[0][0]
    axes[1].scatter(rbs_df[best_feat], rbs_df['F_on_OD'], alpha=0.7, color='coral')
    axes[1].set_xlabel(best_feat)
    axes[1].set_ylabel('RBS Strength (F-on/OD)')
    axes[1].set_title(f'{best_feat} vs Strength')

# Scatter of sd_present vs Strength
bp = axes[2].boxplot([rbs_df[rbs_df['sd_present']==0]['F_on_OD'],
                      rbs_df[rbs_df['sd_present']==1]['F_on_OD']])
axes[2].set_xticklabels(['No SD', 'SD Found'])
axes[2].set_ylabel('RBS Strength (F-on/OD)')
axes[2].set_title('SD Motif Presence vs Strength')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/plots/rbs_feature_analysis_fixed.png', dpi=150)
print(f"✓ Saved: {OUTPUT_DIR}/plots/rbs_feature_analysis_fixed.png")
plt.show()

In [ ]:
# ============================================================
# 8. SUMMARY STATISTICS
# ============================================================
print("\n" + "=" * 70)
print("SD MOTIF DETECTION SUMMARY (FIXED)")
print("=" * 70)

sd_found = rbs_df[rbs_df['sd_present'] == 1]
sd_not_found = rbs_df[rbs_df['sd_present'] == 0]

print(f"\nSD motif found: {len(sd_found)} / {len(rbs_df)} ({len(sd_found)/len(rbs_df)*100:.1f}%)")
print(f"SD motif NOT found: {len(sd_not_found)} / {len(rbs_df)} ({len(sd_not_found)/len(rbs_df)*100:.1f}%)")

if len(sd_found) > 0:
    print(f"\nAverage strength (SD found): {sd_found['F_on_OD'].mean():.1f}")
    print(f"Average strength (NO SD): {sd_not_found['F_on_OD'].mean():.1f}")

# Show strong RBS with SD status
print(f"\nTop 5 Strongest RBS:")
for idx, row in rbs_df.nlargest(5, 'F_on_OD')[['strain', 'F_on_OD', 'sd_present', 'sd_mismatches', 'sd_best_score']].iterrows():
    sd_status = "✓" if row['sd_present'] == 1 else "✗"
    print(f"  Strain {int(row['strain'])}: Strength={row['F_on_OD']:.1f}, SD={sd_status}, Score={row['sd_best_score']:.2f}")

print("\n" + "=" * 70)
print("RBS FEATURE ENGINEERING FIXED - COMPLETE!")
print("=" * 70)
print(f"\nResults saved to: {OUTPUT_DIR}/data/")
print(f"  - rbs_features_fixed.csv")
print(f"  - rbs_correlations_fixed.csv")
print(f"Plots saved to: {OUTPUT_DIR}/plots/")